# Project 8 – Plane dataset: image chips (OSS4008A)
**Notebook 03:** cuts square images ("chips") of Ontario airports from the open Ontario aerial imagery, ready to label with LabelMe.

**How it works**
1. Finds the **aprons** (where planes park) at each airport using OpenStreetMap, so we don't cut hundreds of empty grass chips.
2. Downloads the Ontario imagery tiles over those aprons at about **0.2 m per pixel** and stitches them into **1024 × 1024 px** chips (about 215 m × 215 m each).
3. Saves the chips and a table of where each chip is (`chips_index.csv`) to your Google Drive, and gives you a zip to download for labelling.

**Data:** Ontario Imagery Web Map Service, Land Information Ontario (Open Government Licence – Ontario).
Apron locations: © OpenStreetMap contributors (ODbL), used only to choose where to cut.

**How to run:** Runtime → Run all. Cell 6 asks for permission to use your Google Drive.

In [ ]:
# Cell 1 – Libraries (all pre-installed in Colab)
import io, math, os, time, shutil
import numpy as np
import pandas as pd
import requests
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

In [ ]:
# Cell 2 – Settings

# Ontario aerial imagery tiles (~20 cm). ArcGIS orders the tile address as z/y/x.
ONTARIO_TILES = ('https://ws.lioservices.lrc.gov.on.ca/arcgis2/rest/services/'
                 'LIO_Imagery/Ontario_Imagery_Web_Map_Service/MapServer/tile/{z}/{y}/{x}')
ZOOM = 19                 # zoom 19 ≈ 0.21 m per pixel in southern Ontario (close to the imagery's own 20 cm)
CHIP_TILES = 4            # 4 x 4 tiles of 256 px = 1024 x 1024 px chips (~215 m across)
MAX_CHIPS_PER_AIRPORT = 40

# Ontario airports with lots of parked planes: big airliners, regional turboprops and small private planes
AIRPORTS = ['YYZ', 'YOW', 'YTZ', 'YHM', 'YKF', 'YXU', 'YOO']

# Airport locations: the cleaned table from notebook 01, in the course GitHub repository
CSV_URL = ('https://raw.githubusercontent.com/AdityaR28/'
           'OSS4008A---Introduction-to-Remote-Sensing-and-Applications/main/'
           'students/nifemi_ogunmeru/data/canada_airports_clean.csv')
airports = pd.read_csv(CSV_URL)
chosen = airports[airports['iata_code'].isin(AIRPORTS)].set_index('iata_code').loc[AIRPORTS]
chosen[['name', 'municipality', 'lat', 'lon']]

## Helper functions
Tile maths (Web Mercator), stitching tiles into chips, and checks for empty images.

In [ ]:
# Cell 3 – Helper functions
import math, io
import numpy as np
from PIL import Image

TILE_PX = 256                       # every web-map tile is 256 x 256 pixels


def lonlat_to_tile(lon, lat, z):
    """Which web-map tile (x, y) contains this point at zoom z (standard Web Mercator tiling)."""
    n = 2 ** z
    x = int((lon + 180.0) / 360.0 * n)
    lat_r = math.radians(lat)
    y = int((1.0 - math.log(math.tan(lat_r) + 1.0 / math.cos(lat_r)) / math.pi) / 2.0 * n)
    return x, y


def tile_to_lonlat(x, y, z):
    """Longitude/latitude of the TOP-LEFT corner of tile (x, y) at zoom z."""
    n = 2 ** z
    lon = x / n * 360.0 - 180.0
    lat = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * y / n))))
    return lon, lat


def metres_per_pixel(lat, z):
    """Ground size of one pixel at this latitude and zoom (Web Mercator)."""
    return 156543.03392 * math.cos(math.radians(lat)) / (2 ** z)


def chips_for_boxes(boxes, z, chip_tiles):
    """All chips (aligned to a fixed grid of chip_tiles x chip_tiles tiles) that touch any of the boxes.
    boxes: list of (west, south, east, north) in degrees. Returns a sorted list of (x0, y0) top-left tiles."""
    chips = set()
    for w, s, e, n in boxes:
        x_min, y_min = lonlat_to_tile(w, n, z)          # top-left of the box
        x_max, y_max = lonlat_to_tile(e, s, z)          # bottom-right of the box
        for cx in range(x_min // chip_tiles, x_max // chip_tiles + 1):
            for cy in range(y_min // chip_tiles, y_max // chip_tiles + 1):
                chips.add((cx * chip_tiles, cy * chip_tiles))
    return sorted(chips)


def chip_bounds(x0, y0, z, chip_tiles):
    """(west, south, east, north) of a chip whose top-left tile is (x0, y0)."""
    w, n = tile_to_lonlat(x0, y0, z)
    e, s = tile_to_lonlat(x0 + chip_tiles, y0 + chip_tiles, z)
    return w, s, e, n


def build_chip(x0, y0, z, chip_tiles, fetch_tile):
    """Stitch chip_tiles x chip_tiles tiles into one image.
    fetch_tile(z, x, y) returns PNG/JPEG bytes, or None if the tile is missing.
    Returns (PIL image, fraction of tiles that were missing)."""
    size = TILE_PX * chip_tiles
    chip = Image.new("RGB", (size, size), (128, 128, 128))      # grey where a tile is missing
    missing = 0
    for i in range(chip_tiles):
        for j in range(chip_tiles):
            data = fetch_tile(z, x0 + i, y0 + j)
            if data is None:
                missing += 1
                continue
            try:
                tile = Image.open(io.BytesIO(data)).convert("RGB")
            except Exception:
                missing += 1
                continue
            chip.paste(tile, (i * TILE_PX, j * TILE_PX))
    return chip, missing / chip_tiles ** 2


def looks_empty(img, min_std=8.0):
    """True if the image is (almost) one flat colour: no imagery there (blank or 'no data' tiles)."""
    return float(np.asarray(img.convert("L"), dtype=np.float32).std()) < min_std


def apron_boxes_from_overpass(osm_json):
    """Bounding boxes (west, south, east, north) of the apron polygons in an Overpass 'out geom' answer."""
    boxes = []
    for el in osm_json.get("elements", []):
        pts = el.get("geometry") or []
        if len(pts) < 3:
            continue
        lons = [p["lon"] for p in pts]
        lats = [p["lat"] for p in pts]
        boxes.append((min(lons), min(lats), max(lons), max(lats)))
    return boxes


def box_around(lon, lat, half_km):
    """A square box of +/- half_km around a point (fallback when no apron data is found)."""
    dlat = half_km / 111.32
    dlon = half_km / (111.32 * math.cos(math.radians(lat)))
    return (lon - dlon, lat - dlat, lon + dlon, lat + dlat)


def spread_sample(items, k):
    """Keep at most k items, evenly spread through the list (so a cap doesn't keep only one corner)."""
    if len(items) <= k:
        return list(items)
    step = len(items) / k
    return [items[int(i * step)] for i in range(k)]

In [ ]:
# Cell 4 – Test: download ONE tile at Ottawa airport and show it (checks the imagery service and zoom level)
def fetch_tile(z, x, y, tries=2):
    """Download one imagery tile; returns the image bytes, or None if it is missing."""
    url = ONTARIO_TILES.format(z=z, x=x, y=y)
    for _ in range(tries):
        try:
            r = requests.get(url, timeout=20)
            if r.status_code == 200 and r.headers.get('Content-Type', '').startswith('image'):
                return r.content
            if r.status_code == 404:
                return None
        except requests.RequestException:
            pass
        time.sleep(1)
    return None

yow = chosen.loc['YOW']
tx, ty = lonlat_to_tile(yow['lon'], yow['lat'], ZOOM)
data = fetch_tile(ZOOM, tx, ty)
print('Tile downloaded ✅' if data else '❌ No tile at this zoom – tell Claude')
print(f'Pixel size here: {metres_per_pixel(yow["lat"], ZOOM):.2f} m')
Image.open(io.BytesIO(data)) if data else None

In [ ]:
# Cell 5 – Find the aprons (where planes park) at each airport, and list the chips that cover them
OVERPASS = 'https://overpass-api.de/api/interpreter'

def apron_boxes(lat, lon, radius_m=3000):
    """Bounding boxes of OpenStreetMap aprons within radius_m of the airport point."""
    query = f'[out:json][timeout:60];way["aeroway"="apron"](around:{radius_m},{lat},{lon});out geom;'
    for _ in range(3):
        try:
            r = requests.post(OVERPASS, data={'data': query}, timeout=90)
            if r.status_code == 200:
                return apron_boxes_from_overpass(r.json())
        except requests.RequestException:
            pass
        time.sleep(5)                       # the public server is busy sometimes: wait and retry
    return []

plan = {}
for code_, a in chosen.iterrows():
    boxes = apron_boxes(a['lat'], a['lon'])
    source = f'{len(boxes)} aprons'
    if not boxes:                           # fallback: a 1.4 km square around the airport point
        boxes, source = [box_around(a['lon'], a['lat'], 0.7)], 'no apron data – square around airport'
    chips = spread_sample(chips_for_boxes(boxes, ZOOM, CHIP_TILES), MAX_CHIPS_PER_AIRPORT)
    plan[code_] = chips
    print(f'{code_}: {len(chips):3d} chips  ({source})')
    time.sleep(2)                           # be polite to the free Overpass server

print('Total chips to download:', sum(len(c) for c in plan.values()))

In [ ]:
# Cell 6 – Download and save the chips to Google Drive (takes a few minutes)
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/oss4008a_planes'
IMG_DIR = f'{OUT}/images'
os.makedirs(IMG_DIR, exist_ok=True)

def make_one(job):
    """Build, check and save one chip. Returns a row for the index table, or None if skipped."""
    code_, (x0, y0) = job
    name = f'{code_}_{ZOOM}_{x0}_{y0}.jpg'
    img, missing = build_chip(x0, y0, ZOOM, CHIP_TILES, fetch_tile)
    if missing > 0.25 or looks_empty(img):  # skip chips with no imagery
        return None
    img.save(f'{IMG_DIR}/{name}', quality=90)
    w, s, e, n = chip_bounds(x0, y0, ZOOM, CHIP_TILES)
    return {'file': name, 'airport': code_, 'zoom': ZOOM, 'x0': x0, 'y0': y0,
            'west': w, 'south': s, 'east': e, 'north': n,
            'm_per_px': metres_per_pixel((s + n) / 2, ZOOM)}

jobs = [(code_, xy) for code_, chips in plan.items() for xy in chips]
rows = []
start = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:    # 8 downloads at a time
    for i, row in enumerate(pool.map(make_one, jobs), 1):
        if row:
            rows.append(row)
        if i % 20 == 0 or i == len(jobs):
            print(f'{i}/{len(jobs)} chips processed, {len(rows)} saved ({time.time() - start:.0f} s)')

index = pd.DataFrame(rows)
index.to_csv(f'{OUT}/chips_index.csv', index=False)    # where each chip is on Earth (used later for the map)
print('\nSaved', len(index), 'chips to', IMG_DIR)
index.groupby('airport').size()

In [ ]:
# Cell 7 – Preview 6 chips (one per airport where possible)
import matplotlib.pyplot as plt
sample = index.groupby('airport').head(1).head(6)
fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 4))
for ax, (_, r) in zip(np.atleast_1d(axes), sample.iterrows()):
    ax.imshow(Image.open(f'{IMG_DIR}/{r["file"]}'))
    ax.set_title(r['file'], fontsize=8)
    ax.axis('off')
plt.show()

In [ ]:
# Cell 8 – Zip the images and download them for labelling in LabelMe
shutil.make_archive('/content/plane_chips_images', 'zip', IMG_DIR)
print(f'Zip size: {os.path.getsize("/content/plane_chips_images.zip") / 1e6:.0f} MB')
from google.colab import files
files.download('/content/plane_chips_images.zip')